# *PRÁCTICA 4.- Modelado no supervisado*

## *Paso 1: Extracción de datos*

### *Se descargaron los datos abiertos del sistema ECOBICI desde el año 2010 hasta 2025, lo que equivale a un total aproximado de entre 90 y 100 millones de registros. Todos estos datos se integraron en un único dataframe.*
### *Para este paso, se desarrolló un programa al que se le pasaban los archivos de datos, y que se encargaba de limpiarlos. Durante este proceso se realizaron las siguientes acciones:*

### *Se eliminaron las columnas de fechas, ya que no eran necesarias y solo ocupaban espacio.*

### *Se convirtió el campo Genero_Usuario a tipo entero para optimizar el uso de memoria, asignando 0 para mujeres y 1 para hombres.*

### *Se corrigieron valores numéricos mal formateados, como aquellos con ceros a la izquierda (por ejemplo, "001" se cambió por "1"), ya que causaban problemas al tener distintos tipos de datos en la misma columna.*

### *Para los casos donde aparecían valores como "123-125", se decidió conservar únicamente el primer número ("123"). Esta decisión se tomó tras consultar con ECOBICI, quienes indicaron que el primer valor representa el nombre del módulo y el segundo la estación.*

### *Una vez limpios, todos los archivos se unieron en un solo archivo CSV. Aunque en el ejemplo inicial se usaron solo dos archivos, este proceso se aplicó a los 180 archivos disponibles para consolidarlos en un único archivo general.*

In [ ]:
import pandas as pd
import numpy as np
import os
from tkinter import Tk
from tkinter.filedialog import askopenfilenames
import re
from datetime import datetime

def select_csv_files():
    """
    para abrir archivos
    """
    Tk().withdraw()  
    file_paths = askopenfilenames(
        title="Seleccione los archivos",
        filetypes=[("Archivos CSV", "*.csv"), ("Todos los archivos", "*.*")]
    )
    return file_paths

def extract_date_from_filename(filename):
    """
    extrae el año y mes de cada archivo
    """
    try:
        base = os.path.basename(filename).split('.')[0]
        year, month = map(int, base.split('-'))
        return datetime(year=year, month=month, day=1)
    except:
        return datetime.min  

def analyze_csv(file_path):
    """
    Analizando el csv
    """
    if not file_path:
        print("No se seleccionó ningún archivo.")
        return None
    
    try:
        df = pd.read_csv(file_path)
        
        print(f"\n===== INFORMACIÓN DEL ARCHIVO =====")
        print(f"Archivo: {os.path.basename(file_path)}")
        print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
        
        print("\n----- Nombres de columnas -----")
        for i, col in enumerate(df.columns):
            print(f"{i+1}. {col}")

        print("\n----- Tipos de datos y valores nulos -----")
        for col in df.columns:
            null_count = df[col].isna().sum()
            null_percent = (null_count / len(df)) * 100
            print(f"{col}: Tipo={df[col].dtype}, Nulos={null_count} ({null_percent:.2f}%)")
        
        print("\n----- Muestra aleatoria de datos (5 registros) -----")
        print(df.sample(5, random_state=42))
        
        return df
    
    except Exception as e:
        print(f"Error al leer el archivo: {e}")
        return None

def rename_columns(df):
    """
    renombramos las columnas
    """
    columnas_correctas = [
        "Genero_Usuario", "Edad_Usuario", "Bici",
        "Ciclo_Estacion_Retiro", "Fecha_Retiro", "Hora_Retiro",
        "Ciclo_Estacion_Arribo", "Fecha_Arribo", "Hora_Arribo"
    ]
    
    if len(df.columns) > len(columnas_correctas):
        print(f"\nADVERTENCIA: El archivo tiene {len(df.columns)} columnas. Eliminando columnas...")
        df = df.iloc[:, :len(columnas_correctas)]
    
    df.columns = columnas_correctas[:len(df.columns)]
    
    print("\n----- Columnas renombradas -----")
    for i, col in enumerate(df.columns):
        print(f"{i+1}. {col}")
    
    return df

def clean_station_ids(value):
    """
    se limpuano los id para evitar nulos
    """
    if pd.isna(value) or value == "" or value is None:
        return -1
    

    str_value = str(value).strip()
    

    if str_value == "":
        return -1
    
    if "-" in str_value:
        parts = str_value.split("-")
        for part in parts:
            if part.strip().isdigit():
                return int(part.strip())
        return -1
    
    try:
        return int(float(str_value))  
    except (ValueError, TypeError):
        numbers = re.findall(r'\d+', str_value)
        if numbers:
            return int(numbers[0])
        return -1

def clean_bike_ids(value):
    """
    se limpian id de bicis de los casos null o 002 
    """
    if pd.isna(value) or value == "" or value is None:
        return -1
    
    str_value = str(value).strip()
    
    if str_value == "" or str_value.lower() == "temporal":
        return -1
    
    try:
        return int(float(str_value))  
    except (ValueError, TypeError):
        return str_value

def clean_gender(df):
    """
    se convierten a valores numericos para reducir memoria
    """
    df = df.copy()
    
    mode_gender = df["Genero_Usuario"].mode()[0]
    df.loc[df["Genero_Usuario"].isna(), "Genero_Usuario"] = mode_gender
    
    gender_series = df["Genero_Usuario"].astype(str).str.strip().str.upper()
    

    gender_map = {
        'F': 0,
        'M': 1
    }
    
    df["Genero_Usuario"] = gender_series.map(lambda x: gender_map.get(x, 2))
    print("\nDistribución de valores de género después de la limpieza:")
    print(df["Genero_Usuario"].value_counts().sort_index())
    print("0: Femenino, 1: Masculino, 2: Otro/Desconocido")
    
    return df

def clean_age(df):
    """
    si hay nulo lo que tomamos es la media
    """
    median_age = df["Edad_Usuario"].median()
    df = df.copy()
    df.loc[df["Edad_Usuario"].isna(), "Edad_Usuario"] = median_age
    return df

def clean_data(df):
    """
aqui se limpia todo
    """
    print("\n===== LIMPIEZA DE DATOS =====")
    
    cleaned_df = df.copy()
    
    print("Limpiando IDs ...")
    cleaned_df["Ciclo_Estacion_Retiro"] = cleaned_df["Ciclo_Estacion_Retiro"].apply(
        lambda x: clean_station_ids(x) if pd.isna(x) or str(x).strip() == "" or not str(x).isdigit() else int(float(x)))
    cleaned_df["Ciclo_Estacion_Arribo"] = cleaned_df["Ciclo_Estacion_Arribo"].apply(
        lambda x: clean_station_ids(x) if pd.isna(x) or str(x).strip() == "" or not str(x).isdigit() else int(float(x)))
    
    print("Limpiando IDs de bicicleta...")
    cleaned_df["Bici"] = cleaned_df["Bici"].apply(
        lambda x: clean_bike_ids(x) if pd.isna(x) or str(x).strip() == "" else x)
    
    print("Gestionando valores nulos de género...")
    cleaned_df = clean_gender(cleaned_df)


    print("Gestionando valores nulos de edad...")
    cleaned_df = clean_age(cleaned_df)
    
    print("Eliminando filas con fechas o horas nulas...")
    before_rows = len(cleaned_df)
    
    cleaned_df = cleaned_df.dropna(subset=["Fecha_Retiro", "Fecha_Arribo", "Hora_Retiro", "Hora_Arribo"])
    
    after_rows = len(cleaned_df)
    dropped_rows = before_rows - after_rows
    print(f"Se eliminaron {dropped_rows} filas con fechas o horas nulas ({dropped_rows/before_rows*100:.2f}% del total)")
    
    print("\nEliminando columnas de fecha (conservando solo las horas)...")
    cleaned_df = cleaned_df.drop(columns=["Fecha_Retiro", "Fecha_Arribo"], errors="ignore")
    
    return cleaned_df

def check_cleaning_results(original_df, cleaned_df):
    
    print("\n===== RESULTADOS DE LIMPIEZA =====")
    
    print(f"Filas originales: {len(original_df)}, Filas después de limpieza: {len(cleaned_df)}")
    
    null_counts = cleaned_df.isna().sum()
    if null_counts.sum() > 0:
        print("\nADVERTENCIA: Todavía hay valores nulos en los datos:")
        for col, count in null_counts.items():
            if count > 0:
                print(f"  - {col}: {count} valores nulos")
    else:
        print("\nÉxito: No hay valores nulos en los datos limpios")
    
    print("\n----- Muestra de datos limpios (5 registros) -----")
    print(cleaned_df.sample(5, random_state=42))
    
    return null_counts.sum() == 0 

def save_cleaned_data(cleaned_df, original_file_path):
    """
    guarda cada archivo pero con el sufijo clean
    """
    file_name, file_ext = os.path.splitext(original_file_path)
    cleaned_file_path = f"{file_name}_Clean{file_ext}"
    
    cleaned_df.to_csv(cleaned_file_path, index=False)
    print(f"\nDatos limpios guardados en: {cleaned_file_path}")
    return cleaned_file_path

def combine_dataframes(file_paths):
    """
    Combina los archivos limpiados
    """
    cleaned_dfs = []
    
    for file_path in file_paths:
        clean_path = f"{os.path.splitext(file_path)[0]}_Clean.csv"
        if os.path.exists(clean_path):
            df = pd.read_csv(clean_path)
            cleaned_dfs.append((extract_date_from_filename(clean_path), df))
    
    if not cleaned_dfs:
        print("\nNo se encontraron archivos limpios para combinar.")
        return None
    
   
    cleaned_dfs.sort(key=lambda x: x[0])
    combined_df = pd.concat([df for (date, df) in cleaned_dfs], ignore_index=True)
    

    combined_path = "datos_combinados.csv"
    combined_df.to_csv(combined_path, index=False)
    print(f"\n===== ARCHIVOS COMBINADOS =====")
    print(f"Se combinaron {len(cleaned_dfs)} archivos en: {combined_path}")
    print(f"Dimensiones finales: {combined_df.shape[0]} filas x {combined_df.shape[1]} columnas")
    
    return combined_path

def process_file(file_path):
    """
    Columnas
    """
    print(f"\n=== PROCESANDO ARCHIVO: {os.path.basename(file_path)} ===")
    
    original_df = analyze_csv(file_path)
    if original_df is None:
        return False
    
    df = rename_columns(original_df)
    
    cleaned_df = clean_data(df)
    
    success = check_cleaning_results(original_df, cleaned_df)
    
    if success:
        save_cleaned_data(cleaned_df, file_path)
    else:
        print("\nADVERTENCIA: La limpieza no eliminó todos los valores nulos.")
        save_anyway = input("¿Desea guardar el archivo de todos modos? (s/n): ")
        if save_anyway.lower() == 's':
            save_cleaned_data(cleaned_df, file_path)
    
    return True

def main():
    print("=== LIMPIADOR DE DATOS DE BICICLETAS ===")
    print("Seleccione uno o más archivos CSV para procesar...")
    
    file_paths = select_csv_files()
    if not file_paths:
        return
    
    for file_path in file_paths:
        process_file(file_path)

    combine_dataframes(file_paths)

if __name__ == "__main__":
    main()

: 

Después de procesar todos los archivos con el código que desarrollamos, logramos generar un archivo CSV que agrupa datos históricos limpios, abarcando desde 2010 hasta 2025. Este archivo es bastante extenso, con más de 120 millones de registros. Sin embargo, nos percatamos de que trabajar directamente con el CSV provocaba fallos frecuentes en mi computadora. Por esta razón, decidimos convertirlo a formato Parquet, ya que es más compacto y optimizado para las operaciones que realizaremos a continuación.

In [ ]:
import pandas as pd
import dask.dataframe as dd
from tkinter import Tk, filedialog
from tkinter.filedialog import askopenfilename
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def select_csv_file():
    """Selecciona archivo"""
    Tk().withdraw()
    return askopenfilename(
        title="Se selecciona archivo de ECOBICI",
        filetypes=[("CSV", "*.csv"), ("Todos", "*.*")]
    )

def convert_to_parquet(csv_path):
    """Convertimos a parquet"""
    print("Leyendo archivo CSV...")
    ddf = dd.read_csv(csv_path, assume_missing=True)
    
    parquet_path = csv_path.replace('.csv', '.parquet')
    print(f"Convirtiendo a Parquet en: {parquet_path}")
    ddf.to_parquet(parquet_path, engine='pyarrow')
    print("✅ Conversión completada")
    return parquet_path

def create_tad(parquet_path):
    """Crea la Tabla"""
    print(" Creando Tabla Analítica de Datos (TAD)...")
    ddf = dd.read_parquet(parquet_path)
    
    if 'Hora_Retiro' in ddf.columns:
        ddf['Hora_Retiro'] = ddf['Hora_Retiro'].str.split(':').str[0].astype(float)
    if 'Hora_Arribo' in ddf.columns:
        ddf['Hora_Arribo'] = ddf['Hora_Arribo'].str.split(':').str[0].astype(float)
    
    columns_to_keep = [
        'Genero_Usuario', 
        'Edad_Usuario', 
        'Bici', 
        'Ciclo_Estacion_Retiro',
        'Hora_Retiro',
        'Ciclo_Estacion_Arribo',
        'Hora_Arribo'
    ]
    
    tad = ddf[columns_to_keep]
    print("✅ TAD creada exitosamente")
    return tad

def train_models(tad, sample_size=0.1):
    """Entrena modelos de clustering y muestra resultados ."""
    print(" Entrenando modelos de clustering...")
    
    sample = tad.sample(frac=sample_size, random_state=42).compute()
    print("\n Muestra de la TAD:")
    print(sample.head())
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(sample.select_dtypes(include=['number']))
    
    print("\n K-Means clustering:")
    kmeans = MiniBatchKMeans(n_clusters=5, batch_size=1000, random_state=42)
    kmeans.fit(scaled_data)
    print("  - Inercia:", kmeans.inertia_)
    print("  - Conteo de puntos por cluster:")
    print(pd.Series(kmeans.labels_).value_counts().sort_index())
    
    print("\n Gaussian Mixture Model:")
    gmm = GaussianMixture(n_components=5, random_state=42)
    gmm.fit(scaled_data)
    print("  - Log-likelihood:", gmm.lower_bound_)
    print("  - Conteo de puntos por cluster:")
    gmm_labels = gmm.predict(scaled_data)
    print(pd.Series(gmm_labels).value_counts().sort_index())
    
    print("\n Agglomerative Clustering:")
    agg = AgglomerativeClustering(n_clusters=5)
    agg_labels = agg.fit_predict(scaled_data[:10000])
    print("  - Conteo de puntos por cluster:")
    print(pd.Series(agg_labels).value_counts().sort_index())
    
    return {
        'kmeans': kmeans,
        'gmm': gmm,
        'agg': agg,
        'sample': sample,
        'scaled_data': scaled_data
    }

def profile_by_time(sample, models):
    """Realiza perfilamiento por tiempo."""
    print(" Realizando perfilamiento...")
    
    sample['kmeans_cluster'] = models['kmeans'].labels_
    
    time_profile = sample.groupby(['Hora_Retiro', 'kmeans_cluster']).size().unstack()
    
    return time_profile

def plot_time_profile(time_profile):
    """Visualiza el perfil de clusters."""
    plt.figure(figsize=(12, 6))
    time_profile.fillna(0).plot(kind='bar', stacked=True)
    plt.title('Distribución de Clusters por Hora del Día')
    plt.xlabel('Hora del día')
    plt.ylabel('Número de viajes')
    plt.legend(title='Cluster')
    plt.show()

def plot_cluster_centers(centers, feature_names):
    """Visualiza los centroides."""
    plt.figure(figsize=(12, 6))
    for i in range(centers.shape[0]):
        plt.plot(centers[i], label=f'Cluster {i}')
    plt.xticks(range(len(feature_names)), feature_names, rotation=45)
    plt.title('Centroides de Clusters (K-Means)')
    plt.xlabel('Características')
    plt.ylabel('Valor escalado')
    plt.legend()
    plt.grid()
    plt.show()

def main():
    csv_path = select_csv_file()
    if not csv_path:
        print("❌ No se seleccionó ningún archivo")
        return
    
    parquet_path = convert_to_parquet(csv_path)
    
    tad = create_tad(parquet_path)
    
    models = train_models(tad, sample_size=0.05)
    
    time_profile = profile_by_time(models['sample'], models)
    
    print("\n=== RESULTADOS ===")
    print("\nDistribución de clusters:")
    print(time_profile.head(24)) 
    
    print("\nCentroides de clusters:")
    print(models['kmeans'].cluster_centers_)
    
+    print("\n Generando visualizaciones...")
    
    plot_time_profile(time_profile)
    
    feature_names = models['sample'].select_dtypes(include=['number']).columns.tolist()
    plot_cluster_centers(models['kmeans'].cluster_centers_, feature_names)

if __name__ == "__main__":
    main()

: 

*Análisis del uso de bicicletas por grupo a lo largo del día*

*Nuestros hallazgos principales son:*

*El grupo 3 es el más activo durante las horas de la mañana (entre las 6 a.m. y la 1 p.m.). Esto sugiere que estos usuarios podrían ser personas que se desplazan a sus trabajos o centros de estudio.*
*Los grupos 0 y 4 predominan durante las tardes y noches (de 2 p.m. a 9 p.m.). Es probable que estos usuarios estén regresando a casa o participando en actividades de ocio.*
*El grupo 1 se destaca por su actividad en las primeras horas de la madrugada (de 12 a.m. a 6 a.m.). Esto podría indicar un perfil de usuario que utiliza el servicio en horarios menos comunes.*
*Características distintivas de cada grupo*

*Nuestros hallazgos principales son:*

*El grupo 2 se caracteriza por un uso excepcionalmente alto de bicicletas (variable "Bici"). Esto podría señalar a usuarios con comportamientos muy específicos o que utilizan las bicicletas de forma intensiva.*
*Los grupos 1 y 4 muestran valores negativos significativos en la variable "Genero_Usuario". Esto podría deberse a usuarios que no proporcionaron su género o que representan una minoría.*
*El grupo 3 tiende a utilizar el servicio temprano en el día, ya que sus valores en "Hora_Retiro" y "Hora_Arribo" son bajos.*
*El grupo 0 está compuesto principalmente por usuarios con una edad promedio, que son predominantemente hombres y que realizan sus viajes en las horas de la tarde.*
*En resumen, el sistema ECOBICI revela patrones de uso claramente distintos según la hora del día.*